# Colab preflight (T0)

Purpose: verify that this Colab runtime can be the project's measurement platform **before** any real benchmark runs.
Checks: GPU model, CUDA toolchain, Nsight Compute availability, **ncu performance-counter permission**, and a rough HBM copy bandwidth.

How to use:
1. Runtime → Change runtime type → **A100 GPU**.
2. Runtime → Run all (takes ~3–5 minutes, longer if ncu must be installed).
3. Copy the JSON printed by the **last cell** and paste it back to Claude.

This notebook changes nothing on GitHub and needs no secrets.

In [ ]:
import json, os, re, shutil, subprocess, glob
R = {}  # preflight report
def sh(cmd, timeout=900):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    return p.returncode, (p.stdout + p.stderr).strip()

# 1. GPU model
rc, out = sh("nvidia-smi --query-gpu=name,memory.total,driver_version,clocks.max.sm,clocks.max.mem,power.limit --format=csv,noheader")
print(out)
R["nvidia_smi"] = out
R["gpu_name"] = out.split(",")[0].strip() if rc == 0 else None
R["gpu_ok"] = bool(R["gpu_name"]) and "A100-SXM4-40GB" in R["gpu_name"]
print("GPU OK (A100-SXM4-40GB):", R["gpu_ok"])

In [ ]:
# 2. CUDA toolchain
rc, out = sh("nvcc --version")
print(out)
m = re.search(r"release (\d+)\.(\d+)", out)
R["nvcc"] = f"{m.group(1)}.{m.group(2)}" if m else None
rc, out = sh("nvidia-smi | head -4"); print(out)

In [ ]:
# 3. Find Nsight Compute; install it from the CUDA apt repo if missing
def find_ncu():
    c = [shutil.which("ncu")] + sorted(glob.glob("/usr/local/cuda*/bin/ncu")) + sorted(glob.glob("/opt/nvidia/nsight-compute/*/ncu"))
    return next((x for x in c if x and os.path.exists(x)), None)

ncu = find_ncu()
R["ncu_preinstalled"] = ncu is not None
if ncu is None and R["nvcc"]:
    pkg = "cuda-nsight-compute-" + R["nvcc"].replace(".", "-")
    print("ncu not found; installing", pkg)
    rc, out = sh(f"apt-get update -qq && apt-get install -y -qq {pkg}", timeout=1800)
    print(out[-2000:])
    R["ncu_install"] = {"pkg": pkg, "rc": rc}
    ncu = find_ncu()
R["ncu_path"] = ncu
if ncu:
    rc, out = sh(f"{ncu} --version"); print(out); R["ncu_version"] = out.splitlines()[-1] if out else None
else:
    print("ncu NOT available")

In [ ]:
%%writefile /content/pf_copy.cu
// Minimal copy kernel: rough HBM bandwidth + target for the ncu permission test.
#include <cstdio>
#include <cuda_runtime.h>
__global__ void copy16(const double2* __restrict__ a, double2* __restrict__ b, size_t n) {
    for (size_t i = blockIdx.x * (size_t)blockDim.x + threadIdx.x; i < n; i += (size_t)gridDim.x * blockDim.x)
        b[i] = a[i];
}
int main() {
    const size_t bytes = 1ull << 31;               // 2 GiB per buffer, >> 40 MB L2
    const size_t n = bytes / sizeof(double2);
    double2 *a, *b;
    if (cudaMalloc(&a, bytes) || cudaMalloc(&b, bytes)) { printf("alloc failed\n"); return 1; }
    cudaMemset(a, 0, bytes);
    int sms; cudaDeviceGetAttribute(&sms, cudaDevAttrMultiProcessorCount, 0);
    dim3 grid(sms * 8), block(256);
    for (int i = 0; i < 3; ++i) copy16<<<grid, block>>>(a, b, n);
    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);
    const int iters = 20;
    cudaEventRecord(s);
    for (int i = 0; i < iters; ++i) copy16<<<grid, block>>>(a, b, n);
    cudaEventRecord(e); cudaEventSynchronize(e);
    float ms; cudaEventElapsedTime(&ms, s, e);
    cudaError_t err = cudaGetLastError();
    printf("copy_GBs=%.1f err=%s\n", 2.0 * bytes * iters / (ms * 1e-3) / 1e9, cudaGetErrorString(err));
    return err != cudaSuccess;
}

In [ ]:
# 4. Build + rough HBM copy bandwidth (no ncu)
rc, out = sh("cd /content && nvcc -O3 -arch=sm_80 pf_copy.cu -o pf_copy && ./pf_copy")
print(out)
m = re.search(r"copy_GBs=([\d.]+)", out)
R["rough_hbm_copy_GBs"] = float(m.group(1)) if m else None

In [ ]:
# 5. THE key check: can ncu read hardware performance counters here?
if R.get("ncu_path"):
    cmd = (f"cd /content && {R['ncu_path']} --clock-control none --launch-count 1 "
           "--metrics gpu__time_duration.sum,dram__bytes_read.sum,dram__bytes_write.sum,lts__t_bytes.sum ./pf_copy")
    rc, out = sh(cmd)
    print(out[-3000:])
    R["ncu_rc"] = rc
    R["ncu_perm_error"] = "ERR_NVGPUCTRPERM" in out
    vals = {k: v for k, v in re.findall(r"(gpu__time_duration\.sum|dram__bytes_read\.sum|dram__bytes_write\.sum|lts__t_bytes\.sum)\s+(\S+\s+\S+)", out)}
    R["ncu_metrics_sample"] = vals
    R["ncu_counters_ok"] = (not R["ncu_perm_error"]) and "dram__bytes_read.sum" in vals
else:
    R["ncu_counters_ok"] = False
print("ncu counters OK:", R["ncu_counters_ok"])

In [ ]:
# 6. Environment facts for the plan (read-only queries)
rc, out = sh("nvidia-smi -q -d CLOCK,PERFORMANCE,ECC | head -80"); R["clock_ecc"] = out[:3000]
rc, out = sh("nproc; free -g | head -2; df -h /content | tail -1; python3 --version"); R["host"] = out
rc, out = sh("id -u"); R["uid"] = out
print(R["host"])

In [ ]:
# 7. SUMMARY — copy everything below and paste it back to Claude
short = {k: R.get(k) for k in ["gpu_name","gpu_ok","nvcc","ncu_preinstalled","ncu_install","ncu_path","ncu_version",
                               "rough_hbm_copy_GBs","ncu_rc","ncu_perm_error","ncu_counters_ok","ncu_metrics_sample","uid","host"]}
print(json.dumps(short, indent=1, ensure_ascii=False))